In [ ]:
# fix imports
import os
import sys

module_path = os.path.abspath(os.path.join(".."))
if module_path not in sys.path:
    sys.path.append(module_path)

In [ ]:
import torch
from transformers import AutoTokenizer
from src.eval.beaver import AutoModelForScore

model_name = "PKU-Alignment/beaver-7b-v1.0-cost"

model = AutoModelForScore.from_pretrained(
    model_name,
    device_map="cuda:0",
    torch_dtype=torch.bfloat16,
    max_model_length=2048,
).eval()

tokenizer = AutoTokenizer.from_pretrained(model_name)

In [ ]:
import torch
import torch_tensorrt

model = torch.compile(model, backend="tensorrt")

In [ ]:
BEAVER_PROMPT = """BEGINNING OF CONVERSATION: USER: {behavior} ASSISTANT: {generation}"""

data = [
    ("I like to play football.", "I enjoy playing soccer with my friends."),
    ("I love reading books.", "I am fond of novels and literature."),
]

texts = [BEAVER_PROMPT.format(behavior=pair[0], generation=pair[1]) for pair in data]

In [ ]:
input_ids = tokenizer(
    texts,
    return_tensors="pt",
    padding=True,
).to(model.device)

with torch.inference_mode():
    outputs = model(**input_ids)
    
scores = outputs.end_scores
print(scores)

In [ ]:
input_ids = tokenizer(
    texts,
    return_tensors="pt",
    padding=True,
).to(model.device)

with torch.inference_mode():
    outputs = model(**input_ids)
    
scores = outputs.end_scores
print(scores)

In [ ]:
import torch

torch._dynamo.list_backends()
torch._inductor.list_options()
torch._inductor.config

['TYPE_CHECKING',
 'inplace_padding',
 'can_inplace_pad_graph_input',
 'enable_auto_functionalized_v2',
 'debug',
 'disable_progress',
 'verbose_progress',
 'fx_graph_cache',
 'fx_graph_remote_cache',
 'bundle_triton_into_fx_graph_cache',
 'autotune_local_cache',
 'autotune_remote_cache',
 'bundled_autotune_remote_cache',
 'force_disable_caches',
 'sleep_sec_TESTING_ONLY',
 'custom_op_default_layout_constraint',
 'triton_kernel_default_layout_constraint',
 'cpp_wrapper',
 'online_softmax',
 'dce',
 'static_weight_shapes',
 'size_asserts',
 'nan_asserts',
 'scalar_asserts',
 'pick_loop_orders',
 'inplace_buffers',
 'allow_buffer_reuse',
 'memory_planning',
 'use_fast_math',
 'memory_pool',
 'benchmark_harness',
 'epilogue_fusion',
 'prologue_fusion',
 'epilogue_fusion_first',
 'pattern_matcher',
 'b2b_gemm_pass',
 'post_grad_custom_pre_pass',
 'post_grad_custom_post_pass',
 'joint_custom_pre_pass',
 'joint_custom_post_pass',
 'pre_grad_custom_pass',
 '_pre_fusion_custom_pass',
 'split_c